# CNN + Bidirectional Mamba on LungHist700, with a patient-level split
### Second dataset for the thesis: a real, leakage-free test on patients the model has never seen

**What this notebook does, in one run:**

1. **Pre-trains** the same CNN + bidirectional Mamba model on LC25000 (all 25,000 images, 5 classes).
   LC25000 is used only as a source of general histology features. None of its images are tested here.
2. **Fine-tunes** it on **LungHist700** for 3 classes: adenocarcinoma (`aca`), normal lung (`nor`)
   and squamous cell carcinoma (`scc`).
3. Tests it in two ways, both split **by patient**, so no patient is in both train and test:
   * **the authors' official split** (direct comparison with the published ResNet-50 result), and
   * **5-fold patient-level cross-validation** (every patient is tested once; gives mean ± std).
4. Compares three models under exactly the same folds:
   `cnn_mamba_lcpre` (ours, LC25000-pretrained), `cnn_mamba_scratch` (same model, random start)
   and `resnet50_imagenet` (the architecture the dataset authors used).
5. Runs the **same XAI comparison** as the LC25000 notebook (influence_cc vs Grad-CAM, LIME and
   MambaLRP, with paired Wilcoxon tests and Holm correction) on out-of-fold test images.

### Why this dataset matters

LC25000 was built by augmenting about 1,250 original images, and it has no patient IDs. So even
a careful split can only remove *image* duplicates, not *patient* overlap. LungHist700
(Diosdado et al., *Scientific Data*, 2024, CC BY 4.0) gives a **patient ID for every image**.
A patient-level split is the standard that clinical reviewers expect.

### Facts about LungHist700 used below (from the paper and the authors' code)

| item | value |
|---|---|
| images | 691 in the paper (the metadata file lists 692; the notebook reports any missing file) |
| patients | 45 |
| image size | 1200 × 1600 px, H&E stain |
| microscope / camera | Leica DM 2000 / Leica ICC50 W |
| magnifications | 20x and 40x (trained separately, as the authors did) |
| classes | aca and scc in three differentiation grades (well / moderately / poorly), and normal |
| authors' split | patient-wise, about 80 / 10 / 10 |
| authors' best result, 20x | ResNet-50 (ImageNet), accuracy 0.90, ROC-AUC 0.98 |

### What you must attach before running

* **LungHist700.** Add the Kaggle dataset `abdullahhasansajjad/lunghist700` (it contains
  `data/data.csv` and the original `.jpg` files). Any copy that keeps the original file names works.
* **LC25000.** The same dataset you used before (the one with folders `lung_aca`, `lung_scc`, ...).
* **Internet ON.** It is needed to download the ImageNet weights for ResNet-50.

Run once with `QUICK_TEST = True` (a few minutes), then set it to `False` and use
**Save Version → Save & Run All (Commit)**.

## Section 1 — Configuration

Every setting is in this cell.

* `RESOLUTION = "20x"` matches the authors' main result. You can run `"40x"` as a second experiment.
* Images are shrunk from 1200 × 1600 to **300 × 400**, the same size the authors used. The model
  then sees **224 × 224 crops**: a random crop for training, and six crops that cover the whole
  image for testing (their predictions are averaged).
* Fine-tuning uses a fixed number of epochs with a cosine schedule, and the **last** epoch is kept.
  The validation patients are only watched, never used to pick a checkpoint. With only about
  five validation patients, picking the "best" epoch would mostly pick noise.

In [1]:
QUICK_TEST = False          # True = few minutes on tiny settings. Run this FIRST.
                           # False = the real run (about 5-7 hours on a T4)

# ---------------- LungHist700 ----------------
RESOLUTION   = "20x"       # "20x" (authors' main result) or "40x"
LH_CLASSES   = ["aca", "nor", "scc"]     # sorted order, the same as the authors' code
NUM_CLASSES  = 3
LOAD_H, LOAD_W = 300, 400  # authors' working size (1200x1600 / 4)
IMG_SIZE     = 224
N_FOLDS      = 5
SEED         = 42
MODELS       = ["cnn_mamba_lcpre", "cnn_mamba_scratch", "resnet50_imagenet"]
MAIN_MODEL   = "cnn_mamba_lcpre"

# ---------------- LC25000 pre-training ----------------
LC_CLASSES          = ["lung_aca", "lung_scc", "lung_n", "colon_aca", "colon_n"]
PRETRAIN_EPOCHS     = 1 if QUICK_TEST else 6
PRETRAIN_MAX_PER_CLASS = 40 if QUICK_TEST else 5000
PRETRAIN_LR         = 3e-4

# ---------------- fine-tuning ----------------
FT_EPOCHS    = 2 if QUICK_TEST else 40
BATCH_SIZE   = 8 if QUICK_TEST else 16
LR           = {"cnn_mamba_lcpre": 2e-4, "cnn_mamba_scratch": 3e-4, "resnet50_imagenet": 1e-4}
WEIGHT_DECAY = 0.05
LABEL_SMOOTH = 0.05

# ---------------- architecture (identical to the LC25000 notebook) ----------------
D_MODEL, DEPTH, D_STATE, D_CONV, EXPAND, PATCH = 192, 4, 8, 4, 2, 2
CHUNK_L        = 32
USE_CHECKPOINT = True

# ---------------- XAI ----------------
XAI_PER_CLASS_PER_FOLD = 1 if QUICK_TEST else 8     # up to 3 x 8 x 5 = 120 test images
LIME_SAMPLES   = 40 if QUICK_TEST else 150
DELETION_STEPS = 10 if QUICK_TEST else 25
TOP_FRAC       = 0.20
N_BOOTSTRAP    = 200 if QUICK_TEST else 10000
REFERENCE      = "influence_cc"
ALPHA          = 0.05

OUT_DIR = "/kaggle/working/results_lunghist700"
FIGSHARE_URL = "https://ndownloader.figshare.com/files/45206104"   # LungHist700.rar (670 MB)

print(f"QUICK_TEST={QUICK_TEST} resolution={RESOLUTION} folds={N_FOLDS} "
      f"pretrain_epochs={PRETRAIN_EPOCHS} ft_epochs={FT_EPOCHS}")
if not QUICK_TEST:
    print("\nrough wall-clock on a Tesla T4:")
    print("   LC25000 pre-training, 6 epochs          ~1.8 h")
    print("   fine-tuning, 3 models x 6 runs          ~2.0 h")
    print("   XAI on ~120 images (LIME dominates)     ~1.0 h")
    print("   total                                   ~5 h   (Kaggle limit is 12 h)")

QUICK_TEST=False resolution=20x folds=5 pretrain_epochs=6 ft_epochs=40

rough wall-clock on a Tesla T4:
   LC25000 pre-training, 6 epochs          ~1.8 h
   fine-tuning, 3 models x 6 runs          ~2.0 h
   XAI on ~120 images (LIME dominates)     ~1.0 h
   total                                   ~5 h   (Kaggle limit is 12 h)


## Section 2 — Imports, seeds, GPU

In [2]:
import os, math, random, time, warnings, glob, copy
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
from collections import defaultdict

import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(OUT_DIR, exist_ok=True)
def free_gpu():
    if DEVICE == "cuda": torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
def gpu_peak_gb():
    return torch.cuda.max_memory_allocated()/1024**3 if DEVICE == "cuda" else 0.0

print("torch:", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda": print("gpu  :", torch.cuda.get_device_name(0))
print("output:", OUT_DIR)

torch: 2.10.0+cu128 | device: cuda
gpu  : Tesla T4
output: /kaggle/working/results_lunghist700


## Section 3 — Load LungHist700

The notebook looks for `data.csv` anywhere under `/kaggle/input`, then finds every image by its
file name. The file name is built exactly as in the authors' code:

`{superclass}[_{subclass}]_{resolution}_{image_id}.jpg`, for example `aca_bd_20x_0.jpg` or `nor_20x_5.jpg`.

If the dataset is not attached, the notebook tries to download the official archive from
figshare (a `.rar` file), which needs Internet ON and an extraction tool.

The checks below stop the run if a class is missing or if too many images cannot be found.
The table also shows **how many patients each class has**. This is the real sample size for a
patient-level study, and it is much smaller than the image count.

In [3]:
import subprocess, shutil, urllib.request

def _find_lh_csv(root="/kaggle/input"):
    for p in glob.glob(f"{root}/**/data.csv", recursive=True):
        try:
            cols = set(pd.read_csv(p, nrows=2).columns)
        except Exception:
            continue
        if {"superclass", "resolution", "image_id", "patient_id"} <= cols:
            return p
    return None

def _download_lh(dst="/kaggle/working/lunghist700_raw"):
    os.makedirs(dst, exist_ok=True)
    rar = f"{dst}/LungHist700.rar"
    if not os.path.exists(rar):
        print("downloading LungHist700 from figshare (670 MB) ...")
        urllib.request.urlretrieve(FIGSHARE_URL, rar)
    for tool, args in [("unrar", ["x", "-o+", rar, dst + "/"]), ("7z", ["x", "-y", f"-o{dst}", rar]),
                       ("bsdtar", ["-xf", rar, "-C", dst])]:
        if shutil.which(tool):
            subprocess.run([tool] + args, check=True, stdout=subprocess.DEVNULL); return dst
    raise RuntimeError("The .rar archive could not be extracted (no unrar / 7z / bsdtar). "
                       "Attach the Kaggle dataset 'abdullahhasansajjad/lunghist700' instead.")

LH_CSV = _find_lh_csv()
if LH_CSV is None:
    LH_CSV = _find_lh_csv(_download_lh())
assert LH_CSV, "LungHist700 data.csv not found"
LH_ROOT = os.path.dirname(os.path.dirname(LH_CSV))
print("metadata:", LH_CSV)

# index every jpg once by its base name (search the whole input tree, the layout varies by mirror)
_search_roots = {"/kaggle/input", LH_ROOT}
jpg_index = {}
for r in _search_roots:
    for p in glob.glob(f"{r}/**/*.jp*g", recursive=True):
        jpg_index.setdefault(os.path.basename(p), p)

meta = pd.read_csv(LH_CSV)
meta["superclass"] = meta["superclass"].astype(str).str.strip().str.lower()
has_sub = "subclass" in meta.columns
def _fname(r):
    css = r.superclass if (not has_sub or pd.isnull(r.subclass)) else f"{r.superclass}_{r.subclass}"
    return f"{css}_{r.resolution}_{r.image_id}.jpg"
meta["filename"] = meta.apply(_fname, axis=1)
meta["grade"] = meta.apply(lambda r: r.superclass if (not has_sub or pd.isnull(r.subclass))
                           else f"{r.superclass}_{r.subclass}", axis=1)
meta["path"] = meta.filename.map(jpg_index)

print(f"rows in data.csv: {len(meta)}  |  image files found: {int(meta.path.notna().sum())}")
missing = meta[meta.path.isna()]
if len(missing):
    print("rows whose image file is missing (dropped):")
    print(missing[["filename", "patient_id"]].to_string(index=False))
_res = meta[meta.resolution == RESOLUTION]
assert len(_res) > 0, f"no {RESOLUTION} rows in data.csv"
assert _res.path.notna().mean() > 0.95, f"More than 5% of the {RESOLUTION} images are missing: wrong dataset attached?"

df = meta[meta.path.notna() & (meta.resolution == RESOLUTION)].copy()
assert set(df.superclass) == set(LH_CLASSES), f"unexpected classes: {sorted(set(df.superclass))}"
df["y"] = df.superclass.map({c: i for i, c in enumerate(LH_CLASSES)})
df["patient_id"] = df["patient_id"].astype(int)
df = df.sort_values(["superclass", "grade", "patient_id", "image_id"]).reset_index(drop=True)

summary = (df.groupby("grade").agg(images=("path", "size"), patients=("patient_id", "nunique")))
print(f"\n{RESOLUTION}: {len(df)} images from {df.patient_id.nunique()} patients")
print(summary.to_string())
pc = df.groupby("superclass").agg(images=("path", "size"), patients=("patient_id", "nunique"))
print("\nper class:\n" + pc.to_string())
multi = df.groupby("patient_id").superclass.nunique()
print(f"\npatients with images of more than one class: {int((multi > 1).sum())} "
      f"(e.g. tumour and normal tissue from the same person)")
print("-> the split must be by patient, otherwise the same person's tissue is in train and test.")
summary.to_csv(f"{OUT_DIR}/tableA_dataset_summary.csv")

metadata: /kaggle/input/datasets/abdullahhasansajjad/lunghist700/LungHist700_combined/data/data.csv
rows in data.csv: 691  |  image files found: 691

20x: 359 images from 38 patients
        images  patients
grade                   
aca_bd      57         8
aca_md      44         8
aca_pd      45         7
nor         85        11
scc_bd      50         8
scc_md      30         7
scc_pd      48         6

per class:
            images  patients
superclass                  
aca            146        16
nor             85        11
scc            128        18

patients with images of more than one class: 7 (e.g. tumour and normal tissue from the same person)
-> the split must be by patient, otherwise the same person's tissue is in train and test.


### Loading images into memory

There are only a few hundred images, so each one is decoded **once**, shrunk to 300 × 400, and
kept in RAM. This makes every epoch fast.

If an image is taller than it is wide, it is rotated by 90° first. Rotation does not change the
diagnosis of a tissue section (there is no "up" on a slide), so this is safe, and it avoids
squashing the cells.

**Training augmentation** uses only changes that do not alter the biology: random crop,
horizontal and vertical flips, 90° rotations, and a small change in brightness and colour
(different labs stain slightly differently). Colour is never changed strongly: in H&E,
colour carries real information (haematoxylin = nuclei, eosin = cytoplasm and stroma).

In [4]:
import torchvision.transforms as T
import torchvision.transforms.functional as TF
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def _load_small(p):
    im = Image.open(p).convert("RGB")
    if im.height > im.width: im = im.rotate(90, expand=True)
    return np.asarray(im.resize((LOAD_W, LOAD_H), Image.BILINEAR), dtype=np.uint8)

t0 = time.time()
IMGS = np.stack([_load_small(p) for p in df.path])
print(f"cached {IMGS.shape} uint8 in {time.time()-t0:.0f}s  ({IMGS.nbytes/1e6:.0f} MB)")

# six 224x224 crops that together cover the whole 300x400 image
_ys, _xs = [0, LOAD_H-IMG_SIZE], [0, (LOAD_W-IMG_SIZE)//2, LOAD_W-IMG_SIZE]
CROPS = [(y, x) for y in _ys for x in _xs]
_norm = T.Normalize(MEAN, STD)
_jitter = T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.02)

class LHDataset(Dataset):
    """train=True: one random augmented crop. train=False: the six fixed crops, stacked."""
    def __init__(self, idx, train):
        self.idx, self.train = np.asarray(idx), train
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        x = torch.from_numpy(IMGS[j]).permute(2, 0, 1).float()/255.0
        y = int(df.y.iat[j])
        if self.train:
            top = random.randint(0, LOAD_H-IMG_SIZE); left = random.randint(0, LOAD_W-IMG_SIZE)
            x = x[:, top:top+IMG_SIZE, left:left+IMG_SIZE]
            if random.random() < 0.5: x = x.flip(2)
            if random.random() < 0.5: x = x.flip(1)
            x = torch.rot90(x, random.randint(0, 3), (1, 2))
            x = _jitter(x)
            return _norm(x), y
        return torch.stack([_norm(x[:, a:a+IMG_SIZE, b:b+IMG_SIZE]) for a, b in CROPS]), y

def center_crop_tensor(j):
    """One centred 224x224 crop, used for the XAI evaluation."""
    x = torch.from_numpy(IMGS[j]).permute(2, 0, 1).float()/255.0
    a, b = (LOAD_H-IMG_SIZE)//2, (LOAD_W-IMG_SIZE)//2
    return _norm(x[:, a:a+IMG_SIZE, b:b+IMG_SIZE])

_x, _ = LHDataset([0], False)[0]
assert _x.shape == (len(CROPS), 3, IMG_SIZE, IMG_SIZE)
_x, _ = LHDataset([0], True)[0]
assert _x.shape == (3, IMG_SIZE, IMG_SIZE)
print("datasets ready:", len(CROPS), "test crops per image")

cached (359, 300, 400, 3) uint8 in 17s  (129 MB)
datasets ready: 6 test crops per image


## Section 4 — The model (identical to the LC25000 notebook)

The selective scan and the CNN + bidirectional Mamba model are copied **unchanged** from the
LC25000 notebook, including the check that the fast chunked scan equals the slow sequential one.

In [5]:
def _shift_right(x, n, fill):
    if n >= x.shape[2]: return torch.full_like(x, fill)
    return torch.cat([torch.full_like(x[:,:,:n], fill), x[:,:,:-n]], dim=2)

def _inclusive_scan(a, x):
    """T_j = a_j * T_{j-1} + x_j, in log2(L) vectorised steps."""
    l, off = a.shape[2], 1
    while off < l:
        x = x + a*_shift_right(x, off, 0.0)
        a = a*_shift_right(a, off, 1.0)
        off *= 2
    return x

def selective_scan_parallel(u, delta, A, B, C, D, chunk=None):
    """u,delta:(b,d,l)  A:(d,n)  B,C:(b,n,l)  D:(d,) -> y:(b,d,l)
    Parallel inside a chunk, sequential across chunks. Mathematically identical
    to the full scan; the carry exp(cumsum(delta*A)) is exact."""
    chunk = chunk or CHUNK_L
    b, d, l = u.shape
    log_a = delta.unsqueeze(-1)*A[:,None,:]
    x_in  = delta.unsqueeze(-1)*B.transpose(1,2).unsqueeze(1)*u.unsqueeze(-1)
    h_prev, ys = u.new_zeros(b,d,A.shape[-1]), []
    for s0 in range(0, l, chunk):
        s1 = min(s0+chunk, l); la = log_a[:,:,s0:s1]
        h = _inclusive_scan(la.exp(), x_in[:,:,s0:s1]) + la.cumsum(2).exp()*h_prev.unsqueeze(2)
        ys.append(torch.einsum("bdln,bnl->bdl", h, C[:,:,s0:s1]))
        h_prev = h[:,:,-1]
    return torch.cat(ys,2) + u*D[None,:,None]

def selective_scan_sequential(u, delta, A, B, C, D):
    """Slow reference. Only used to verify the fast path."""
    b, d, l = u.shape
    a = torch.exp(delta.unsqueeze(-1)*A[:,None,:])
    x = delta.unsqueeze(-1)*B.transpose(1,2).unsqueeze(1)*u.unsqueeze(-1)
    h = u.new_zeros(b, d, A.shape[-1]); ys = []
    for t in range(l):
        h = a[:,:,t]*h + x[:,:,t]
        ys.append(torch.einsum("bdn,bn->bd", h, C[:,:,t]))
    return torch.stack(ys,-1) + u*D[None,:,None]

# ---- verification --------------------------------------------------------
torch.manual_seed(0)
_u  = torch.randn(2,6,64); _dl = F.softplus(torch.randn(2,6,64))
_A  = -torch.rand(6,4)-0.1; _B = torch.randn(2,4,64)
_C  = torch.randn(2,4,64);  _D = torch.randn(6)
_ref = selective_scan_sequential(_u,_dl,_A,_B,_C,_D)
for _ck in [1,4,7,32,1000]:
    _e = (selective_scan_parallel(_u,_dl,_A,_B,_C,_D,chunk=_ck)-_ref).abs().max().item()
    print(f"  chunk={_ck:<5d} max |chunked - sequential| = {_e:.2e}")
    assert _e < 1e-4, f"Chunked scan wrong at chunk={_ck} - do not trust any result."
print("OK: chunked scan verified against the sequential reference.")

  chunk=1     max |chunked - sequential| = 0.00e+00
  chunk=4     max |chunked - sequential| = 1.91e-06
  chunk=7     max |chunked - sequential| = 1.91e-06
  chunk=32    max |chunked - sequential| = 1.91e-06
  chunk=1000  max |chunked - sequential| = 2.38e-06
OK: chunked scan verified against the sequential reference.


In [6]:
def _silu_lrp(x):
    """Eq. 4: SiLU with the sigmoid detached, restoring relevance conservation."""
    return x * torch.sigmoid(x).detach()


class MambaBlock(nn.Module):
    def __init__(self, d_model, d_state=D_STATE, d_conv=D_CONV, expand=EXPAND):
        super().__init__()
        d_inner = expand*d_model
        self.d_inner, self.d_state, self.d_conv = d_inner, d_state, d_conv
        self.in_proj = nn.Linear(d_model, d_inner*2, bias=False)
        self.conv1d  = nn.Conv1d(d_inner, d_inner, d_conv, groups=d_inner, padding=d_conv-1)
        self.dt_rank = math.ceil(d_model/16)
        self.x_proj  = nn.Linear(d_inner, self.dt_rank+2*d_state, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, d_inner, bias=True)
        dt = torch.exp(torch.rand(d_inner)*(math.log(0.1)-math.log(1e-3))+math.log(1e-3)).clamp(min=1e-4)
        with torch.no_grad(): self.dt_proj.bias.copy_(dt + torch.log(-torch.expm1(-dt)))
        A = torch.arange(1, d_state+1, dtype=torch.float32).repeat(d_inner,1)
        self.A_log = nn.Parameter(torch.log(A))      # A kept negative -> stable decay
        self.D = nn.Parameter(torch.ones(d_inner))
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)

    def forward(self, x, capture=False):
        b, l, _ = x.shape
        u, z = self.in_proj(x).chunk(2,-1)
        u = F.silu(self.conv1d(u.transpose(1,2))[..., :l])
        x_dbl = self.x_proj(u.transpose(1,2))                    # delta, B, C come FROM u
        delta, B, C = torch.split(x_dbl, [self.dt_rank, self.d_state, self.d_state], -1)
        delta = F.softplus(self.dt_proj(delta)).transpose(1,2)
        B, C = B.transpose(1,2).contiguous(), C.transpose(1,2).contiguous()
        A = -torch.exp(self.A_log.float())
        y = selective_scan_parallel(u, delta, A, B, C, self.D.float())
        gate = F.silu(z.transpose(1,2))
        out = self.out_proj((y*gate).transpose(1,2))
        if capture:
            return out, {"delta":delta.detach(), "A":A.detach(), "B":B.detach(),
                         "C":C.detach(), "u":u.detach(), "gate":gate.detach(),
                         "D":self.D.detach(), "W_out":self.out_proj.weight.detach()}
        return out, None

    def forward_lrp(self, x):
        """MambaLRP forward pass (Jafari et al., NeurIPS 2024).

        Three modifications, exactly as the paper specifies:
          Eq. 4     SiLU        y = x * [sigma(x)]_cst
          Eq. 6-7   SSM         Abar_t, Bbar_t, C_t all treated as constants
          Eq. 9     gate        y = 0.5*(zA*zB) + 0.5*[zA*zB]_cst
        Relevance is then read off as gradient x input. LayerNorm is left standard,
        since the paper's stated rules do not cover it.
        """
        b, l, _ = x.shape
        u, z = self.in_proj(x).chunk(2, -1)
        u = _silu_lrp(self.conv1d(u.transpose(1, 2))[..., :l])
        x_dbl = self.x_proj(u.transpose(1, 2))
        delta, B, C = torch.split(x_dbl, [self.dt_rank, self.d_state, self.d_state], -1)
        # ---- Eq. 6-7: the input-dependent SSM parameters become constants ----
        delta = F.softplus(self.dt_proj(delta)).transpose(1, 2).detach()
        B = B.transpose(1, 2).contiguous().detach()
        C = C.transpose(1, 2).contiguous().detach()
        A = (-torch.exp(self.A_log.float())).detach()
        y = selective_scan_parallel(u, delta, A, B, C, self.D.float())
        # ---- Eq. 9: half-relevance through the multiplicative gate ----
        prod = y * _silu_lrp(z.transpose(1, 2))
        return self.out_proj((0.5*prod + 0.5*prod.detach()).transpose(1, 2))


class BiMambaLayer(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.fwd, self.bwd = MambaBlock(d_model), MambaBlock(d_model)
    def forward(self, x, capture=False):
        h = self.norm(x)
        yf, cf = self.fwd(h, capture); yb, cb = self.bwd(h.flip(1), capture)
        return x + yf + yb.flip(1), ({"fwd":cf,"bwd":cb} if capture else None)
    def forward_lrp(self, x):
        h = self.norm(x)
        return x + self.fwd.forward_lrp(h) + self.bwd.forward_lrp(h.flip(1)).flip(1)


def _ckpt_layer(layer, t): return layer(t, False)[0]

class CNNStem(nn.Module):
    """224 -> 28 via three stride-2 blocks. Local morphology only."""
    def __init__(self, dims=(64,128,128)):
        super().__init__()
        ch, L = (3,)+dims, []
        for i in range(len(dims)):
            L += [nn.Conv2d(ch[i],ch[i+1],3,2,1,bias=False), nn.BatchNorm2d(ch[i+1]), nn.GELU(),
                  nn.Conv2d(ch[i+1],ch[i+1],3,1,1,bias=False), nn.BatchNorm2d(ch[i+1]), nn.GELU()]
        self.net, self.out_dim = nn.Sequential(*L), dims[-1]
    def forward(self, x): return self.net(x)

class CNNMamba(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, d_model=D_MODEL, depth=DEPTH):
        super().__init__()
        self.stem = CNNStem()
        self.patch_embed = nn.Conv2d(self.stem.out_dim, d_model, PATCH, PATCH)
        self.grid = IMG_SIZE//8//PATCH               # 224 -> 28 -> 14 (196 tokens, as ViT-B/16)
        self.num_tokens = self.grid**2
        self.layers = nn.ModuleList([BiMambaLayer(d_model) for _ in range(depth)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)
        self.apply(self._init)
    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
    def tokenize(self, x):
        return self.patch_embed(self.stem(x)).flatten(2).transpose(1,2)
    def from_tokens(self, t, capture=False):
        caps = []
        for layer in self.layers:
            if USE_CHECKPOINT and self.training and not capture and t.requires_grad:
                t = torch.utils.checkpoint.checkpoint(_ckpt_layer, layer, t, use_reentrant=False)
            else:
                t, c = layer(t, capture)
                if capture: caps.append(c)
        return self.head(self.norm(t).mean(1)), caps
    def forward(self, x, capture=False):
        lo, caps = self.from_tokens(self.tokenize(x), capture)
        return (lo, caps) if capture else lo
    def from_tokens_lrp(self, t):
        for layer in self.layers:
            t = layer.forward_lrp(t)
        return self.head(self.norm(t).mean(1))

print("parameters:", sum(p.numel() for p in CNNMamba().parameters())/1e6, "M")

parameters: 2.595395 M


## Section 5 — Pre-training on LC25000

LungHist700 is small (a few hundred images at one magnification). A Mamba model trained from
zero on so few images is likely to under-fit or over-fit. So we first train it on all of LC25000
(5 classes) and then use those weights as a starting point.

This is **not** leakage. LC25000 comes from a different institution and different patients, and
no LC25000 image is ever used for testing here. The `cnn_mamba_scratch` model (same network,
random start) shows how much the pre-training actually helps. That is a result in itself.

Note the physical difference: LC25000 tiles and LungHist700 images were taken with different
equipment and at different scales, so a cell does not have the same size in pixels in the two
datasets. Fine-tuning is what adapts the model to the new scale and stain.

In [7]:
def find_class_dirs(root="/kaggle/input"):
    found = {}
    for dp, dns, _ in os.walk(root):
        for d in dns:
            if d in LC_CLASSES and d not in found: found[d] = os.path.join(dp, d)
    return found

lc_dirs = find_class_dirs()
assert len(lc_dirs) == 5, f"LC25000 not found (found {sorted(lc_dirs)}). Attach it via 'Add Input'."
lc_rows = []
for ci, c in enumerate(LC_CLASSES):
    fs = sorted(glob.glob(os.path.join(lc_dirs[c], "*.jpeg")) +
                glob.glob(os.path.join(lc_dirs[c], "*.jpg")))[:PRETRAIN_MAX_PER_CLASS]
    lc_rows += [(f, ci) for f in fs]
    print(f"  {c:10s} {len(fs):5d}")
print("LC25000 images for pre-training:", len(lc_rows))

class LCDataset(Dataset):
    tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                    T.ColorJitter(0.1, 0.1, 0.1), T.ToTensor(), T.Normalize(MEAN, STD)])
    def __len__(self): return len(lc_rows)
    def __getitem__(self, i):
        p, y = lc_rows[i]
        return self.tf(Image.open(p).convert("RGB")), y

nw = 2 if QUICK_TEST else 4
lc_ld = DataLoader(LCDataset(), BATCH_SIZE, shuffle=True, num_workers=nw, pin_memory=True, drop_last=True)

set_seed(SEED); free_gpu()
pre = CNNMamba(num_classes=5).to(DEVICE)
opt = torch.optim.AdamW(pre.parameters(), lr=PRETRAIN_LR, weight_decay=WEIGHT_DECAY)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, PRETRAIN_EPOCHS*len(lc_ld)))
crit = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
for ep in range(PRETRAIN_EPOCHS):
    pre.train(); tot = n = cor = 0; t0 = time.time()
    for x, y in lc_ld:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        out = pre(x); loss = crit(out, y)
        opt.zero_grad(set_to_none=True); loss.backward()
        nn.utils.clip_grad_norm_(pre.parameters(), 1.0); opt.step(); sch.step()
        tot += loss.item()*y.size(0); n += y.size(0); cor += (out.argmax(1) == y).sum().item()
    print(f"  pretrain epoch {ep+1}/{PRETRAIN_EPOCHS}  loss {tot/max(n,1):.4f}  "
          f"train_acc {cor/max(n,1):.4f}  ({time.time()-t0:.0f}s)", flush=True)
PRETRAINED = {k: v.detach().cpu().clone() for k, v in pre.state_dict().items()}
torch.save(PRETRAINED, f"{OUT_DIR}/weights_cnn_mamba_lc25000_pretrain.pt")
del pre, opt, sch, lc_ld; free_gpu()
print("pre-trained weights saved (the 5-class head is replaced before fine-tuning)")

  lung_aca    5000
  lung_scc    5000
  lung_n      5000
  colon_aca   5000
  colon_n     5000
LC25000 images for pre-training: 25000
  pretrain epoch 1/6  loss 0.4632  train_acc 0.8898  (1012s)
  pretrain epoch 2/6  loss 0.3232  train_acc 0.9577  (1013s)
  pretrain epoch 3/6  loss 0.2825  train_acc 0.9754  (1013s)
  pretrain epoch 4/6  loss 0.2623  train_acc 0.9844  (1012s)
  pretrain epoch 5/6  loss 0.2462  train_acc 0.9910  (1013s)
  pretrain epoch 6/6  loss 0.2397  train_acc 0.9946  (1013s)
pre-trained weights saved (the 5-class head is replaced before fine-tuning)


## Section 6 — Training and evaluation tools

* **Class weights.** The classes are not balanced (normal has fewer images at 20x), so the loss
  gives each class the same total weight, as the authors did.
* **Metrics.** Accuracy, **balanced accuracy** (the mean of per-class recall, which is not fooled
  by class imbalance), macro-F1 and one-vs-one macro ROC-AUC (the same AUC the authors report).
* **Every split is checked.** An assertion stops the run if any patient appears in two sets.

In [8]:
import torchvision
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             roc_auc_score, confusion_matrix)
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit

def make_model(name):
    if name == "cnn_mamba_lcpre":
        m = CNNMamba(num_classes=NUM_CLASSES)
        sd = {k: v for k, v in PRETRAINED.items() if not k.startswith("head.")}
        missing, unexpected = m.load_state_dict(sd, strict=False)
        assert set(missing) == {"head.weight", "head.bias"} and not unexpected, (missing, unexpected)
        return m
    if name == "cnn_mamba_scratch":
        return CNNMamba(num_classes=NUM_CLASSES)
    if name == "resnet50_imagenet":
        m = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
        return m
    raise ValueError(name)

def _logits(m, x):
    o = m(x); return o[0] if isinstance(o, tuple) else o

@torch.no_grad()
def predict_proba(m, idx):
    """Mean softmax over the six crops -> (n, 3)."""
    m.eval(); P = []
    ld = DataLoader(LHDataset(idx, False), batch_size=max(1, BATCH_SIZE//2), shuffle=False, num_workers=2)
    for x, _ in ld:
        b, k = x.shape[:2]
        p = F.softmax(_logits(m, x.view(b*k, *x.shape[2:]).to(DEVICE)), 1).view(b, k, -1).mean(1)
        P.append(p.cpu())
    return torch.cat(P).numpy()

def scores(y, p):
    pred = p.argmax(1)
    out = dict(acc=accuracy_score(y, pred), bal_acc=balanced_accuracy_score(y, pred),
               macro_f1=f1_score(y, pred, average="macro", labels=list(range(NUM_CLASSES)), zero_division=0))
    try:    out["auc_ovo"] = roc_auc_score(y, p, multi_class="ovo", average="macro", labels=list(range(NUM_CLASSES)))
    except ValueError: out["auc_ovo"] = float("nan")          # a class is absent from this test set
    return out

def check_disjoint(*parts):
    for i in range(len(parts)):
        for j in range(i+1, len(parts)):
            a = set(df.patient_id.iloc[parts[i]]); b = set(df.patient_id.iloc[parts[j]])
            assert not (a & b), f"patients {sorted(a & b)} appear in two sets: split is broken"

def train_model(name, tr_idx, va_idx, seed):
    set_seed(seed); free_gpu()
    m = make_model(name).to(DEVICE)
    ytr = df.y.to_numpy()[tr_idx]
    cnt = np.bincount(ytr, minlength=NUM_CLASSES).astype(float)
    assert (cnt > 0).all(), f"a class has no training image: {cnt}"
    w = torch.tensor(len(ytr)/(NUM_CLASSES*cnt), dtype=torch.float32, device=DEVICE)
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=LABEL_SMOOTH)
    ld = DataLoader(LHDataset(tr_idx, True), BATCH_SIZE, shuffle=True, num_workers=2,
                    drop_last=len(tr_idx) > BATCH_SIZE, worker_init_fn=lambda k: np.random.seed(seed+k))
    opt = torch.optim.AdamW(m.parameters(), lr=LR[name], weight_decay=WEIGHT_DECAY)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, FT_EPOCHS*len(ld)))
    hist = []
    for ep in range(FT_EPOCHS):
        m.train(); tot = n = 0
        for x, y in ld:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            loss = crit(_logits(m, x), y)
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step(); sch.step()
            tot += loss.item()*y.size(0); n += y.size(0)
        if len(va_idx) and (ep == FT_EPOCHS-1 or (ep+1) % 10 == 0):
            va = scores(df.y.to_numpy()[va_idx], predict_proba(m, va_idx))["acc"]
            hist.append((ep+1, tot/max(n, 1), va))
    return m, hist

print("training tools ready")

training tools ready


## Section 7 — Experiment 1: the authors' official patient split

These are the patient IDs from the authors' own code (`get_reproducible_ids`). Using them makes
our numbers **directly comparable** with the published ResNet-50 result (20x: accuracy 0.90).

Be careful when reading this table: the official test set has only **4 patients at 20x**
(6 at 40x). One patient more or less changes the accuracy by several points. That is why
Section 8 also runs 5-fold cross-validation over all patients.

In [9]:
OFFICIAL = {"20x": dict(train=[2,3,4,5,7,8,12,14,15,16,17,18,20,21,23,24,25,26,28,29,30,33,36,37,38,39,41,42,45],
                        val=[1,6,27,32,44], test=[9,13,31,40]),
            "40x": dict(train=[2,6,8,9,10,12,13,14,16,18,19,21,22,24,28,29,31,33,34,35,36,38,40,44],
                        val=[1,4,17,26,30,37,45], test=[11,15,20,25,32,43])}
ids = OFFICIAL[RESOLUTION]
pid = df.patient_id.to_numpy()
o_tr, o_va, o_te = [np.flatnonzero(np.isin(pid, ids[k])) for k in ("train", "val", "test")]
check_disjoint(o_tr, o_va, o_te)
_unused = sorted(set(pid) - set(ids["train"]) - set(ids["val"]) - set(ids["test"]))
print(f"official split: {len(o_tr)}/{len(o_va)}/{len(o_te)} images, "
      f"{len(ids['train'])}/{len(ids['val'])}/{len(ids['test'])} patients"
      + (f"  (patients not in the official lists, left out: {_unused})" if _unused else ""))
print("test images per class:", np.bincount(df.y.to_numpy()[o_te], minlength=NUM_CLASSES).tolist(), LH_CLASSES)

off_rows = []
for name in MODELS:
    t0 = time.time()
    m, hist = train_model(name, o_tr, o_va, SEED)
    p = predict_proba(m, o_te); s = scores(df.y.to_numpy()[o_te], p)
    off_rows.append(dict(model=name, **{k: round(v, 4) for k, v in s.items()},
                         minutes=round((time.time()-t0)/60, 1)))
    print(f"  {name:20s} acc {s['acc']:.4f}  bal_acc {s['bal_acc']:.4f}  macroF1 {s['macro_f1']:.4f}  "
          f"AUC {s['auc_ovo']:.4f}   val(last) {hist[-1][2] if hist else float('nan'):.3f}", flush=True)
    del m; free_gpu()
off_df = pd.DataFrame(off_rows)
off_df.to_csv(f"{OUT_DIR}/tableB_official_split.csv", index=False)
print("\npublished reference (authors, ResNet-50): 20x acc 0.90 / AUC 0.98;  40x acc 0.82 / AUC 0.94")
print(off_df.to_string(index=False))

official split: 263/48/48 images, 29/5/4 patients
test images per class: [18, 19, 11] ['aca', 'nor', 'scc']
  cnn_mamba_lcpre      acc 0.8333  bal_acc 0.8341  macroF1 0.8333  AUC 0.9460   val(last) 0.729
  cnn_mamba_scratch    acc 0.7917  bal_acc 0.7824  macroF1 0.7794  AUC 0.8930   val(last) 0.500
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 205MB/s]


  resnet50_imagenet    acc 0.9583  bal_acc 0.9512  macroF1 0.9512  AUC 0.9933   val(last) 0.833

published reference (authors, ResNet-50): 20x acc 0.90 / AUC 0.98;  40x acc 0.82 / AUC 0.94
            model    acc  bal_acc  macro_f1  auc_ovo  minutes
  cnn_mamba_lcpre 0.8333   0.8341    0.8333   0.9460      7.5
cnn_mamba_scratch 0.7917   0.7824    0.7794   0.8930      7.5
resnet50_imagenet 0.9583   0.9512    0.9512   0.9933      2.2


## Section 8 — Experiment 2: 5-fold patient-level cross-validation (the main result)

`StratifiedGroupKFold` puts **each patient in exactly one test fold** and keeps the class mix of
the folds as similar as it can. Inside each training part, about 15% of the patients are held
out as validation, for monitoring only.

All three models use the **same folds and the same seeds**, so the comparison is paired.

We report:

* mean ± std over the 5 folds (the usual number), and
* the **pooled out-of-fold** result: every image is predicted by the one model that never saw
  its patient. Its 95% CI comes from a **patient-level bootstrap**: we resample *patients*, not
  images, because images of the same patient are not independent.
* the paired difference between models, with the same patient-level bootstrap.

In [10]:
y_all, pid = df.y.to_numpy(), df.patient_id.to_numpy()
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = []
for f, (trv, te_) in enumerate(sgkf.split(np.zeros(len(df)), y_all, groups=pid)):
    gss = GroupShuffleSplit(1, test_size=0.15, random_state=SEED+f)
    a, b = next(gss.split(trv, y_all[trv], groups=pid[trv]))
    tr_, va_ = trv[a], trv[b]
    check_disjoint(tr_, va_, te_)
    FOLDS.append((tr_, va_, te_))
    print(f"fold {f}: train {len(tr_):3d} img/{len(set(pid[tr_])):2d} pat | val {len(va_):3d}/{len(set(pid[va_])):2d} | "
          f"test {len(te_):3d}/{len(set(pid[te_])):2d}  test classes {np.bincount(y_all[te_], minlength=NUM_CLASSES).tolist()}")
_all_test = np.concatenate([t for _, _, t in FOLDS])
assert len(_all_test) == len(df) and len(set(_all_test)) == len(df), "every image must be tested exactly once"
print("OK: every image is tested exactly once, by a model that never saw its patient.")

OOF = {m: np.zeros((len(df), NUM_CLASSES)) for m in MODELS}
fold_rows, FOLD_STATES = [], {}
for f, (tr_, va_, te_) in enumerate(FOLDS):
    for name in MODELS:
        t0 = time.time()
        m, hist = train_model(name, tr_, va_, SEED + 100*f)
        p = predict_proba(m, te_); OOF[name][te_] = p
        s = scores(y_all[te_], p)
        fold_rows.append(dict(fold=f, model=name, n_test=len(te_), **{k: round(v, 4) for k, v in s.items()},
                              minutes=round((time.time()-t0)/60, 1)))
        print(f"  fold {f} {name:20s} acc {s['acc']:.4f}  bal_acc {s['bal_acc']:.4f}  "
              f"F1 {s['macro_f1']:.4f}  ({(time.time()-t0)/60:.1f} min)", flush=True)
        if name == MAIN_MODEL:
            FOLD_STATES[f] = {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
            torch.save(FOLD_STATES[f], f"{OUT_DIR}/weights_{MAIN_MODEL}_fold{f}.pt")
        del m; free_gpu()

fold_df = pd.DataFrame(fold_rows)
fold_df.to_csv(f"{OUT_DIR}/tableC_cv_per_fold.csv", index=False)
oof_df = df[["filename", "patient_id", "superclass", "grade", "y"]].copy()
for name in MODELS:
    for c, cn in enumerate(LH_CLASSES): oof_df[f"{name}_p_{cn}"] = OOF[name][:, c]
    oof_df[f"{name}_pred"] = OOF[name].argmax(1)
oof_df.to_csv(f"{OUT_DIR}/oof_predictions.csv", index=False)
print("saved per-fold table and out-of-fold predictions")

fold 0: train 250 img/26 pat | val  50/ 5 | test  59/ 7  test classes [22, 0, 37]
fold 1: train 220 img/25 pat | val  65/ 5 | test  74/ 8  test classes [26, 18, 30]
fold 2: train 241 img/26 pat | val  60/ 5 | test  58/ 7  test classes [8, 24, 26]
fold 3: train 229 img/25 pat | val  42/ 5 | test  88/ 8  test classes [40, 16, 32]
fold 4: train 250 img/25 pat | val  29/ 5 | test  80/ 8  test classes [50, 27, 3]
OK: every image is tested exactly once, by a model that never saw its patient.
  fold 0 cnn_mamba_lcpre      acc 0.8475  bal_acc 0.8323  F1 0.5569  (7.1 min)
  fold 0 cnn_mamba_scratch    acc 0.7797  bal_acc 0.7598  F1 0.5081  (7.1 min)
  fold 0 resnet50_imagenet    acc 0.7966  bal_acc 0.8010  F1 0.5262  (2.1 min)
  fold 1 cnn_mamba_lcpre      acc 0.7703  bal_acc 0.7866  F1 0.7700  (6.3 min)
  fold 1 cnn_mamba_scratch    acc 0.6892  bal_acc 0.7182  F1 0.7051  (6.3 min)
  fold 1 resnet50_imagenet    acc 0.7568  bal_acc 0.7761  F1 0.7660  (1.9 min)
  fold 2 cnn_mamba_lcpre      acc 0

In [11]:
def patient_bootstrap(metric_fn, n=N_BOOTSTRAP, seed=SEED):
    """Resample PATIENTS with replacement; images of one patient move together."""
    rng = np.random.default_rng(seed)
    pats = np.unique(pid); where = {p: np.flatnonzero(pid == p) for p in pats}
    vals = []
    for _ in range(n):
        idx = np.concatenate([where[p] for p in rng.choice(pats, size=len(pats), replace=True)])
        vals.append(metric_fn(idx))
    return np.percentile(vals, [2.5, 97.5])

summ = []
for name in MODELS:
    fm = fold_df[fold_df.model == name]
    pooled = scores(y_all, OOF[name])
    lo, hi = patient_bootstrap(lambda i: accuracy_score(y_all[i], OOF[name][i].argmax(1)))
    summ.append(dict(model=name,
                     acc_mean=round(fm.acc.mean(), 4), acc_std=round(fm.acc.std(ddof=1), 4),
                     bal_acc_mean=round(fm.bal_acc.mean(), 4), bal_acc_std=round(fm.bal_acc.std(ddof=1), 4),
                     macro_f1_mean=round(fm.macro_f1.mean(), 4), macro_f1_std=round(fm.macro_f1.std(ddof=1), 4),
                     auc_mean=round(fm.auc_ovo.mean(), 4),
                     pooled_acc=round(pooled["acc"], 4), pooled_acc_ci95=f"[{lo:.3f}, {hi:.3f}]",
                     pooled_bal_acc=round(pooled["bal_acc"], 4), pooled_macro_f1=round(pooled["macro_f1"], 4)))
summ_df = pd.DataFrame(summ)
summ_df.to_csv(f"{OUT_DIR}/tableD_cv_summary.csv", index=False)
pd.set_option("display.width", 220, "display.max_columns", 30)
print("=== 5-fold patient-level CV ===")
print(summ_df.to_string(index=False))

# paired model differences, patient-level bootstrap
diff_rows = []
for other in [m for m in MODELS if m != MAIN_MODEL]:
    d_fn = lambda i: (accuracy_score(y_all[i], OOF[MAIN_MODEL][i].argmax(1))
                      - accuracy_score(y_all[i], OOF[other][i].argmax(1)))
    lo, hi = patient_bootstrap(d_fn)
    d = d_fn(np.arange(len(df)))
    diff_rows.append(dict(main=MAIN_MODEL, other=other, acc_diff=round(d, 4),
                          ci95_low=round(lo, 4), ci95_high=round(hi, 4),
                          verdict=("main better" if lo > 0 else "other better" if hi < 0 else "no clear difference")))
diff_df = pd.DataFrame(diff_rows)
diff_df.to_csv(f"{OUT_DIR}/tableE_model_differences.csv", index=False)
print("\n=== accuracy difference, main model minus other (patient bootstrap 95% CI) ===")
print(diff_df.to_string(index=False))

# confusion matrix and per-grade accuracy of the main model
cm = confusion_matrix(y_all, OOF[MAIN_MODEL].argmax(1), labels=list(range(NUM_CLASSES)))
np.save(f"{OUT_DIR}/cm_{MAIN_MODEL}_oof.npy", cm)
print(f"\nconfusion matrix, {MAIN_MODEL}, out-of-fold (rows = true, cols = predicted):")
print("        " + " ".join(f"{c:>6s}" for c in LH_CLASSES))
for i, r in enumerate(cm): print(f"{LH_CLASSES[i]:>6s}  " + " ".join(f"{v:>6d}" for v in r))
grade = (oof_df.assign(correct=(oof_df[f"{MAIN_MODEL}_pred"] == oof_df.y))
               .groupby("grade").agg(images=("y", "size"), patients=("patient_id", "nunique"),
                                     accuracy=("correct", "mean")))
grade.round(4).to_csv(f"{OUT_DIR}/tableF_accuracy_by_grade.csv")
print("\naccuracy by differentiation grade (main model):")
print(grade.round(4).to_string())

=== 5-fold patient-level CV ===
            model  acc_mean  acc_std  bal_acc_mean  bal_acc_std  macro_f1_mean  macro_f1_std  auc_mean  pooled_acc pooled_acc_ci95  pooled_bal_acc  pooled_macro_f1
  cnn_mamba_lcpre    0.7138   0.0972        0.7186       0.1066         0.6330        0.0959    0.8581      0.7103  [0.622, 0.788]          0.7349           0.7258
cnn_mamba_scratch    0.7131   0.0442        0.7288       0.0282         0.6365        0.1038    0.8277      0.7103  [0.613, 0.786]          0.7432           0.7274
resnet50_imagenet    0.7887   0.0739        0.7571       0.1337         0.6884        0.1574    0.8750      0.7799  [0.693, 0.856]          0.8051           0.7921

=== accuracy difference, main model minus other (patient bootstrap 95% CI) ===
           main             other  acc_diff  ci95_low  ci95_high             verdict
cnn_mamba_lcpre cnn_mamba_scratch    0.0000   -0.0677     0.0704 no clear difference
cnn_mamba_lcpre resnet50_imagenet   -0.0696   -0.1556     0.01

## Section 9 — Explainability on unseen patients

The XAI methods and metrics are copied **unchanged** from the LC25000 notebook:

| method | what it is |
|---|---|
| `influence_cc` | **ours**: class-conditional influence through the frozen Mamba dynamics |
| `influence_cc_last`, `influence_cc_abs` | ablations of ours |
| `influence_rollout` | the class-agnostic version (the "stability trap") |
| `grad_cam`, `lime`, `mamba_lrp` | baselines; MambaLRP is the Mamba-specific state of the art |

Metrics: stability under rotation/noise/brightness (IoU, rotation-aligned, target class fixed),
deletion and insertion AUC (faithfulness), and model-randomisation sanity checks.

**Which images and which model.** From each fold's **test** patients we take up to
`XAI_PER_CLASS_PER_FOLD` images per class, and explain them with **that fold's** model, the one
that never saw those patients. Each image is explained on its **centre 224 × 224 crop**, so the
explanation belongs to one well-defined input.

In [12]:
class InfluenceRollout:
    """Run 1's class-agnostic frozen-dynamics score. Kept as the 'before' baseline."""
    def __init__(self, model): self.model = model
    @staticmethod
    def _dir(cap, wbi=True):
        delta, A, B, u = cap["delta"], cap["A"], cap["B"], cap["u"]
        log_a = delta.unsqueeze(-1)*A[:,None,:]
        cum   = log_a.cumsum(2)
        decay = (2.0*(cum[:,:,-1:,:]-cum)).clamp(min=-60.0).exp()
        gain  = (delta.unsqueeze(-1)*B.transpose(1,2).unsqueeze(1))**2
        if wbi: gain = gain*(u.unsqueeze(-1)**2)
        return (decay*gain).sum(dim=(1,3))
    @staticmethod
    def _norm(s):
        s = s - s.amin(1, keepdim=True)
        return s/(s.amax(1, keepdim=True)+1e-8)
    def per_layer(self, x):
        self.model.eval()
        with torch.no_grad(): _, caps = self.model(x, capture=True)
        return [self._norm(self._dir(c["fwd"]) + self._dir(c["bwd"]).flip(1)) for c in caps]
    def __call__(self, x, residual=0.5):
        maps = self.per_layer(x)
        o = torch.ones_like(maps[0])
        for m in maps: o = o*(residual + (1-residual)*m)
        o = self._norm(o).view(-1,1,self.model.grid,self.model.grid)
        return F.interpolate(o, (IMG_SIZE,IMG_SIZE), mode="bilinear", align_corners=False)
    def spatial_spread(self, x):
        """Mean distance of the influence mass from its own centroid, per layer.
        Growing with depth = the Mamba stack really does widen its context."""
        maps = self.per_layer(x); g = self.model.grid
        yy, xx = torch.meshgrid(torch.arange(g).float(), torch.arange(g).float(), indexing="ij")
        coords = torch.stack([yy.flatten(), xx.flatten()], -1)
        out = []
        for m in maps:
            w = (m/(m.sum(1, keepdim=True)+1e-8)).cpu()
            d = (coords.unsqueeze(0) - (w @ coords).unsqueeze(1)).norm(dim=-1)
            out.append(float((w*d).sum(1).mean()))
        return out


class ClassConditionalInfluence:
    """Forward-pass, class-conditional frozen-dynamics relevance (Section 8)."""
    def __init__(self, model, use_prior=True, residual=0.5, mode="relu"):
        self.model, self.use_prior, self.residual, self.mode = model, use_prior, residual, mode
    @staticmethod
    def _reverse_state_sum(cap):
        """S_s = g_s*C_s + A_{s+1} * S_{s+1}, via one reversed associative scan."""
        a = (cap["delta"].unsqueeze(-1)*cap["A"][:,None,:]).exp()
        X = cap["gate"].unsqueeze(-1)*cap["C"].transpose(1,2).unsqueeze(1)
        return _inclusive_scan(_shift_right(a.flip(2),1,1.0), X.flip(2)).flip(2)
    def _dir(self, cap, a_cls):
        S = self._reverse_state_sum(cap)
        delta, B, u, D, gate = cap["delta"], cap["B"], cap["u"], cap["D"], cap["gate"]
        R = (torch.einsum("bdln,bnl->bdl", S, B)*delta*u*a_cls[:,:,None]).sum(1)
        return R + (D[None,:,None]*u*gate*a_cls[:,:,None]).sum(1)      # skip path
    def _class_vector(self, cap, target):
        """a_c = W_out^T (gamma * W_head[c]) -- linear, no autograd."""
        gamma = self.model.norm.weight.detach()
        return (self.model.head.weight.detach()[target]*gamma) @ cap["W_out"]
    @staticmethod
    def _norm(s):
        s = s - s.amin(1, keepdim=True)
        return s/(s.amax(1, keepdim=True)+1e-8)
    def __call__(self, x, target=None):
        self.model.eval()
        with torch.no_grad():
            logits, caps = self.model(x, capture=True)
            if target is None: target = logits.argmax(1)
            last = caps[-1]
            R = (self._dir(last["fwd"], self._class_vector(last["fwd"], target))
                 + self._dir(last["bwd"], self._class_vector(last["bwd"], target)).flip(1))
            R = self._norm(F.relu(R) if self.mode == "relu" else R.abs())
            if self.use_prior and len(caps) > 1:
                prior = torch.ones_like(R)
                for c in caps[:-1]:
                    m = InfluenceRollout._norm(InfluenceRollout._dir(c["fwd"])
                                               + InfluenceRollout._dir(c["bwd"]).flip(1))
                    prior = prior*(self.residual + (1-self.residual)*m)
                R = self._norm(R*prior)
        g = self.model.grid
        return F.interpolate(R.view(-1,1,g,g), (IMG_SIZE,IMG_SIZE),
                             mode="bilinear", align_corners=False)
print("explainers defined")

explainers defined


In [13]:
def grad_cam(model, x, target=None):
    model.eval()
    t = model.tokenize(x); t.requires_grad_(True)
    logits, _ = model.from_tokens(t, capture=False)
    if target is None: target = logits.argmax(1)
    grad = torch.autograd.grad(logits.gather(1, target[:,None]).sum(), t)[0]
    cam = F.relu((grad*t).sum(-1))
    cam = cam - cam.amin(1, keepdim=True); cam = cam/(cam.amax(1, keepdim=True)+1e-8)
    g = model.grid
    return F.interpolate(cam.view(-1,1,g,g), (IMG_SIZE,IMG_SIZE),
                         mode="bilinear", align_corners=False).detach()

def lime_map(model, x, target=None, n_samples=None, n_segments=50):
    from skimage.segmentation import slic
    from sklearn.linear_model import Ridge
    n_samples = n_samples or LIME_SAMPLES
    model.eval(); outs = []
    with torch.no_grad():
        bl = model(x)
        if isinstance(bl, tuple): bl = bl[0]
    tgt = bl.argmax(1) if target is None else target
    for i in range(x.shape[0]):
        img = x[i:i+1]; arr = img[0].permute(1,2,0).cpu().numpy()
        seg = slic(arr, n_segments=n_segments, compactness=10.0, channel_axis=-1, start_label=0)
        seg_t = torch.from_numpy(seg).to(x.device); k = int(seg.max())+1
        masks = np.random.binomial(1,0.5,(n_samples,k)).astype(np.float32); masks[0] = 1.0
        fill = img.mean(dim=(2,3), keepdim=True); probs = []
        with torch.no_grad():
            for b0 in range(0, n_samples, 16):
                batch = []
                for m in masks[b0:b0+16]:
                    keep = torch.from_numpy(m).to(x.device)[seg_t][None,None]
                    batch.append(img*keep + fill*(1-keep))
                lo = model(torch.cat(batch,0))
                if isinstance(lo, tuple): lo = lo[0]
                probs.append(F.softmax(lo,1)[:, tgt[i]].cpu().numpy())
        w = np.clip(Ridge(alpha=1.0).fit(masks, np.concatenate(probs)).coef_, 0, None)
        m = torch.from_numpy(w).float().to(x.device)[seg_t]
        m = m - m.min(); outs.append((m/(m.max()+1e-8))[None,None])
    return torch.cat(outs, 0)

def mamba_lrp(model, x, target=None):
    """MambaLRP relevance: gradient x input on the conservation-corrected forward pass."""
    model.eval()
    t = model.tokenize(x).detach().requires_grad_(True)
    logits = model.from_tokens_lrp(t)
    if target is None: target = logits.argmax(1)
    grad = torch.autograd.grad(logits.gather(1, target[:, None]).sum(), t)[0]
    R = F.relu((grad * t).sum(-1))                 # positive relevance, as Grad-CAM does
    R = R - R.amin(1, keepdim=True); R = R/(R.amax(1, keepdim=True)+1e-8)
    g = model.grid
    return F.interpolate(R.view(-1,1,g,g), (IMG_SIZE,IMG_SIZE),
                         mode="bilinear", align_corners=False).detach()


def plain_grad_input(model, x, target=None):
    """Gradient x input on the UNMODIFIED forward pass. Only used to prove that the
    MambaLRP detachments actually change something -- see the validation cell."""
    model.eval()
    t = model.tokenize(x).detach().requires_grad_(True)
    logits, _ = model.from_tokens(t, capture=False)
    if target is None: target = logits.argmax(1)
    grad = torch.autograd.grad(logits.gather(1, target[:, None]).sum(), t)[0]
    R = F.relu((grad * t).sum(-1)).detach()
    R = R - R.amin(1, keepdim=True); return R/(R.amax(1, keepdim=True)+1e-8)


# Every method now takes an EXPLICIT target class. This matters: under perturbation
# the prediction can flip, and comparing an explanation for class A against one for
# class B would measure the flip, not the method. The clean image's predicted class
# is therefore fixed and reused everywhere.
# influence_rollout ignores the target by construction -- it is class-agnostic, which
# is precisely the defect the class-conditional variants were built to fix.
METHODS = {
    "influence_rollout": lambda m,x,t: InfluenceRollout(m)(x),
    "influence_cc":      lambda m,x,t: ClassConditionalInfluence(m, use_prior=True)(x, t),
    "influence_cc_last": lambda m,x,t: ClassConditionalInfluence(m, use_prior=False)(x, t),
    "influence_cc_abs":  lambda m,x,t: ClassConditionalInfluence(m, mode="abs")(x, t),
    "grad_cam":          lambda m,x,t: grad_cam(m, x, t),
    "lime":              lambda m,x,t: lime_map(m, x, t),
    "mamba_lrp":         lambda m,x,t: mamba_lrp(m, x, t),
}
print("methods:", list(METHODS))

methods: ['influence_rollout', 'influence_cc', 'influence_cc_last', 'influence_cc_abs', 'grad_cam', 'lime', 'mamba_lrp']


In [14]:
# build the XAI sample: stratified per fold, from test patients only
_rng = np.random.RandomState(SEED)
XAI_ROWS = []                           # (image index, fold)
for f, (_, _, te_) in enumerate(FOLDS):
    for c in range(NUM_CLASSES):
        pos = te_[y_all[te_] == c]
        k = min(XAI_PER_CLASS_PER_FOLD, len(pos))
        XAI_ROWS += [(int(j), f) for j in sorted(_rng.choice(pos, size=k, replace=False))]
xai_idx = np.array([j for j, _ in XAI_ROWS]); xai_fold = np.array([f for _, f in XAI_ROWS])
for j, f in XAI_ROWS:
    assert j in set(FOLDS[f][2]), "an XAI image is not in its fold's test set"
comp = pd.Series(df.superclass.iloc[xai_idx].to_numpy()).value_counts().reindex(LH_CLASSES)
print("XAI sample:", comp.to_dict(), "| total", len(xai_idx))
assert (comp > 0).all(), "a class is missing from the XAI sample"

def load_fold_model(f):
    m = CNNMamba(num_classes=NUM_CLASSES); m.load_state_dict(FOLD_STATES[f]); return m.to(DEVICE).eval()

# MambaLRP validation (same checks as the LC25000 notebook) on fold 0's model
model = load_fold_model(0)
X = torch.stack([center_crop_tensor(j) for j in xai_idx[xai_fold == 0]]).to(DEVICE)
print("validation batch:", tuple(X.shape))

XAI sample: {'aca': 40, 'nor': 32, 'scc': 35} | total 107
validation batch: (16, 3, 224, 224)


In [15]:
from scipy.stats import spearmanr          # (also imported later, harmless)

def _signed_relevance(model, x, target, lrp):
    """Signed token relevance = (grad x input), on the LRP forward pass or the normal one."""
    model.eval()
    t = model.tokenize(x).detach().requires_grad_(True)
    logits = model.from_tokens_lrp(t) if lrp else model.from_tokens(t, capture=False)[0]
    g = torch.autograd.grad(logits.gather(1, target[:, None]).sum(), t)[0]
    return (g * t).sum(-1).detach(), logits.gather(1, target[:, None]).squeeze(1).detach()

_xb = X[:4]
with torch.no_grad():
    _lo = model(_xb)
    if isinstance(_lo, tuple): _lo = _lo[0]
    _tgt = _lo.argmax(1)

# --- check 0a: the LRP forward must give the SAME logits ---------------------
# .detach() changes gradients, never values. If the logits differ, we would be
# explaining a different model than the one we evaluated.
with torch.no_grad():
    _t0 = model.tokenize(_xb)
    _ln, _ = model.from_tokens(_t0.clone(), capture=False)
    _ll    = model.from_tokens_lrp(_t0.clone())
_ferr = (_ln - _ll).abs().max().item()
print(f"check 0a max |normal logits - lrp logits| = {_ferr:.3e}")
assert _ferr < 1e-3, ("The MambaLRP forward pass changes the model's output. "
                      "It would explain a different model -- do not use it.")
print("         the LRP pass alters gradients only, not predictions.")

# --- check 0b: gradient must NOT reach the detached SSM projections -----------
# x_proj and dt_proj feed only delta, B and C. Under Eq. 6-7 those are constants,
# so after a backward through the LRP path their .grad must stay None.
_probe = copy.deepcopy(model)
_tp = _probe.tokenize(_xb).detach().requires_grad_(True)
_probe.from_tokens_lrp(_tp).gather(1, _tgt[:, None]).sum().backward()
_blk = _probe.layers[0].fwd
_leak = (_blk.x_proj.weight.grad is not None) or (_blk.dt_proj.weight.grad is not None)
print(f"check 0b gradient leaked into delta/B/C projections? {_leak}")
assert not _leak, "Eq. 6-7 is not in effect: gradient still flows through Abar/Bbar/C."
print("         Abar, Bbar and C are true constants in the backward pass.")
del _probe, _tp

_R_lrp,   _logit = _signed_relevance(model, _xb, _tgt, lrp=True)
_R_plain, _      = _signed_relevance(model, _xb, _tgt, lrp=False)

# --- check 1: the three detachments must change the attribution ---
_same = torch.allclose(_R_lrp, _R_plain, atol=1e-6)
_reldiff = ((_R_lrp - _R_plain).abs().sum() / (_R_plain.abs().sum() + 1e-12)).item()
print(f"check 1  relative difference from plain grad x input = {_reldiff:.4f}")
assert not _same, ("MambaLRP is identical to plain gradient x input -- "
                   "the detachments are not active. Do not use this baseline.")
print("         the Eq.4 / Eq.6-7 / Eq.9 detachments are active.")

_rhos = [spearmanr(_R_lrp[i].cpu().numpy(), _R_plain[i].cpu().numpy()).correlation
         for i in range(_xb.shape[0])]
_rho = float(np.nanmean(_rhos))
print(f"         Spearman(MambaLRP, plain) = {_rho:.4f}  "
      f"(same family of method, so high is expected; 1.0 would mean no effect)")

# --- check 2: approximate relevance conservation ---
_ratio = (_R_lrp.sum(1) / _logit).cpu().numpy()
print(f"\ncheck 2  sum(relevance)/logit per image = {np.round(_ratio, 3)}")
print(f"         mean {_ratio.mean():.3f}. Not exactly 1: biases (BatchNorm, conv, dt_proj,")
print( "         LayerNorm, the classifier bias) absorb relevance, and the paper's rules")
print( "         do not cover LayerNorm. Quote this number in the paper.")
if QUICK_TEST:
    print("         (QUICK_TEST: the model is barely trained, so its logits are near zero and")
    print("          this ratio is meaningless. Read it from the real run.)")
print("\nMambaLRP port validated -- safe to use as a baseline.")
del _R_lrp, _R_plain, _logit
free_gpu()

check 0a max |normal logits - lrp logits| = 1.192e-07
         the LRP pass alters gradients only, not predictions.
check 0b gradient leaked into delta/B/C projections? False
         Abar, Bbar and C are true constants in the backward pass.
check 1  relative difference from plain grad x input = 0.7756
         the Eq.4 / Eq.6-7 / Eq.9 detachments are active.
         Spearman(MambaLRP, plain) = 0.8124  (same family of method, so high is expected; 1.0 would mean no effect)

check 2  sum(relevance)/logit per image = [0.424 0.141 0.013 0.092]
         mean 0.168. Not exactly 1: biases (BatchNorm, conv, dt_proj,
         LayerNorm, the classifier bias) absorb relevance, and the paper's rules
         do not cover LayerNorm. Quote this number in the paper.

MambaLRP port validated -- safe to use as a baseline.


In [16]:
import torchvision.transforms.functional as TF
from scipy.stats import spearmanr

PERTURB = {"low":dict(angle=5.0,noise=0.02,bright=0.00),
           "mid":dict(angle=10.0,noise=0.05,bright=0.10),
           "high":dict(angle=20.0,noise=0.10,bright=0.20)}

def perturb(x, cfg):
    o = TF.rotate(x, cfg["angle"])
    if cfg["bright"] > 0: o = o*(1.0+cfg["bright"])
    return o + torch.randn_like(o)*cfg["noise"]

def iou_topk(a, b, frac=TOP_FRAC):
    n = a.shape[0]; a, b = a.reshape(n,-1), b.reshape(n,-1)
    k = max(1, int(frac*a.shape[1]))
    ma = torch.zeros_like(a).scatter_(1, a.topk(k,1).indices, 1.0)
    mb = torch.zeros_like(b).scatter_(1, b.topk(k,1).indices, 1.0)
    return ((ma*mb).sum(1)/(((ma+mb)>0).float().sum(1)+1e-8)).cpu().numpy()

def stability(model, fn, x, level, target):
    """IoU between the clean map and the rotation-ALIGNED perturbed map.

    Both maps are computed for the SAME target class (the clean prediction). If the
    perturbed image were allowed to pick its own argmax, a prediction flip would make
    us compare explanations of two different classes -- measuring the flip, not the
    method's stability."""
    cfg = PERTURB[level]
    m0 = fn(model, x, target)
    m1 = TF.rotate(fn(model, perturb(x, cfg), target), -cfg["angle"])
    c = int(IMG_SIZE*0.15)                       # drop borders damaged by rotation
    return iou_topk(m0[...,c:-c,c:-c], m1[...,c:-c,c:-c])

@torch.no_grad()
def prediction_flip_rate(model, x, target, level):
    """How often perturbation changes the predicted class. Reported as a diagnostic:
    a high rate means the perturbation is not 'clinically irrelevant' after all."""
    lo = model(perturb(x, PERTURB[level]))
    if isinstance(lo, tuple): lo = lo[0]
    return (lo.argmax(1) != target).float().cpu().numpy()

@torch.no_grad()
def _curve(model, x, attr, target, steps, mode):
    n = x.shape[0]
    blur = F.conv2d(x, torch.ones(3,1,11,11,device=x.device)/121.0, padding=5, groups=3)
    order = attr.reshape(n,-1).argsort(1, descending=True); total = order.shape[1]
    ps = []
    for i in range(steps+1):
        k = int(total*i/steps)
        if mode == "deletion":
            m = torch.ones(n,total,device=x.device)
            if k: m.scatter_(1, order[:,:k], 0.0)
        else:
            m = torch.zeros(n,total,device=x.device)
            if k: m.scatter_(1, order[:,:k], 1.0)
        m = m.view(n,1,IMG_SIZE,IMG_SIZE)
        lo = model(x*m + blur*(1-m))
        if isinstance(lo, tuple): lo = lo[0]
        ps.append(F.softmax(lo,1).gather(1, target[:,None]).squeeze(1))
    return torch.stack(ps,1)

def sanity_check(model, fn, x, target, scope="full"):
    """Model-parameter randomisation (Adebayo et al.). LOW is good.

    scope="full"  -> CNN stem + patch embedding + every Mamba layer + head.
                     This is the honest 'whole network' test and the one to quote.
    scope="mamba" -> Mamba layers + head only; the CNN stem is left trained.
                     Reported alongside because the earlier version of this study
                     used that narrower scope, and the two are not comparable.

    The target class is held fixed at the clean model's prediction; a randomised
    model's own argmax is meaningless and would confound the comparison.
    """
    m0 = fn(model, x, target)
    rnd = copy.deepcopy(model)
    if scope == "full":
        mods = [rnd.stem, rnd.patch_embed] + list(rnd.layers) + [rnd.head]
    else:
        mods = list(rnd.layers) + [rnd.head]
    for mod in mods:
        for p in mod.parameters(): nn.init.normal_(p, std=0.05)
    m1 = fn(rnd, x, target); del rnd
    return iou_topk(m0, m1)

def true_gradient_map(model, x, target=None):
    model.eval()
    t = model.tokenize(x).detach().requires_grad_(True)
    logits, _ = model.from_tokens(t, capture=False)
    if target is None: target = logits.argmax(1)
    g = torch.autograd.grad(logits.gather(1, target[:,None]).sum(), t)[0]
    s = g.abs().sum(-1); s = s - s.amin(1, keepdim=True)
    return s/(s.amax(1, keepdim=True)+1e-8)

METRICS = ["deletion_auc","insertion_auc","iou_low","iou_mid","iou_high",
           "sanity_iou_full","sanity_iou_mamba","spearman_vs_gradient"]
HIGHER_IS_BETTER = {"deletion_auc":False, "insertion_auc":True, "iou_low":True,
                    "iou_mid":True, "iou_high":True, "sanity_iou_full":False,
                    "sanity_iou_mamba":False, "spearman_vs_gradient":True}
print("metrics:", METRICS)

metrics: ['deletion_auc', 'insertion_auc', 'iou_low', 'iou_mid', 'iou_high', 'sanity_iou_full', 'sanity_iou_mamba', 'spearman_vs_gradient']


In [17]:
# per-image evaluation, fold by fold, each image with the model that never saw its patient
free_gpu()
CHUNK = 4
per_image = {n: defaultdict(list) for n in METHODS}
CURVES_ACC = {n: ([], []) for n in METHODS}
CLS_OF_IMG, IMG_ORDER, FLIP = [], [], {lv: [] for lv in PERTURB}
t_all = time.time()
for f in range(N_FOLDS):
    sel = xai_idx[xai_fold == f]
    if len(sel) == 0: continue
    model = load_fold_model(f)
    X = torch.stack([center_crop_tensor(j) for j in sel]).to(DEVICE)
    with torch.no_grad(): PRED = _logits(model, X).argmax(1)
    CLS_OF_IMG += df.superclass.iloc[sel].tolist(); IMG_ORDER += sel.tolist()
    for lv in PERTURB:
        FLIP[lv].append(np.concatenate([prediction_flip_rate(model, X[i:i+CHUNK], PRED[i:i+CHUNK], lv)
                                        for i in range(0, len(sel), CHUNK)]))
    for name, fn in METHODS.items():
        set_seed(SEED + f)
        for i in range(0, X.shape[0], CHUNK):
            xb, pb = X[i:i+CHUNK], PRED[i:i+CHUNK]
            attr = fn(model, xb, pb)
            dc = _curve(model, xb, attr, pb, DELETION_STEPS, "deletion").cpu().numpy()
            ic = _curve(model, xb, attr, pb, DELETION_STEPS, "insertion").cpu().numpy()
            CURVES_ACC[name][0].append(dc); CURVES_ACC[name][1].append(ic)
            A = per_image[name]
            A["deletion_auc"].append(dc.mean(1)); A["insertion_auc"].append(ic.mean(1))
            for lv in PERTURB: A[f"iou_{lv}"].append(stability(model, fn, xb, lv, pb))
            A["sanity_iou_full"].append(sanity_check(model, fn, xb, pb, "full"))
            A["sanity_iou_mamba"].append(sanity_check(model, fn, xb, pb, "mamba"))
            A["map_std"].append(attr.std(dim=(1, 2, 3)).cpu().numpy())
            g = true_gradient_map(model, xb, pb)
            gg = F.interpolate(g.view(-1, 1, model.grid, model.grid), (IMG_SIZE, IMG_SIZE),
                               mode="bilinear", align_corners=False)
            rho = [spearmanr(attr[j].flatten().cpu().numpy(), gg[j].flatten().cpu().numpy()).correlation
                   for j in range(xb.shape[0])]
            A["spearman_vs_gradient"].append(np.nan_to_num(np.array(rho, dtype=float)))
    print(f"  fold {f}: {len(sel)} images done ({(time.time()-t_all)/60:.1f} min so far)", flush=True)
    del model; free_gpu()

per_image = {n: {k: np.concatenate(v) for k, v in d.items()} for n, d in per_image.items()}
CURVES = {n: (np.concatenate(a).mean(0), np.concatenate(b).mean(0)) for n, (a, b) in CURVES_ACC.items()}
n_img = len(IMG_ORDER)
assert all(len(v) == n_img for d in per_image.values() for v in d.values())

rows = []
for name in METHODS:
    row = {"method": name}; row.update({k: round(float(v.mean()), 4) for k, v in per_image[name].items()})
    if row["map_std"] < 1e-4: print(f"  !! WARNING: {name} gives a near-constant map; ignore its IoU")
    rows.append(row)
xai = pd.DataFrame(rows); xai.to_csv(f"{OUT_DIR}/tableG_xai_comparison.csv", index=False)
recs = [{"image": int(IMG_ORDER[i]), "cls": CLS_OF_IMG[i], "method": nm, "metric": k, "value": float(v)}
        for nm, d in per_image.items() for k, arr in d.items() for i, v in enumerate(arr)]
long_df = pd.DataFrame(recs); long_df.to_csv(f"{OUT_DIR}/per_image_xai_metrics.csv", index=False)
per_cls = (long_df.groupby(["metric", "method", "cls"])["value"].mean().reset_index()
           .pivot_table(index=["metric", "method"], columns="cls", values="value"))
per_cls.round(4).to_csv(f"{OUT_DIR}/tableH_xai_per_class.csv")
flip = {lv: float(np.concatenate(v).mean()) for lv, v in FLIP.items()}
pd.DataFrame([flip]).to_csv(f"{OUT_DIR}/tableI_prediction_flip_rate.csv", index=False)

print(f"\n=== XAI on {n_img} out-of-fold images (unseen patients) ===")
print("iou_* higher = more stable | deletion LOWER = more faithful | insertion HIGHER | sanity_* LOWER")
print(xai.to_string(index=False))
print("\nprediction-flip rate under perturbation:", {k: round(v, 3) for k, v in flip.items()})

  fold 0: 16 images done (6.2 min so far)
  fold 1: 24 images done (15.4 min so far)
  fold 2: 24 images done (24.6 min so far)
  fold 3: 24 images done (33.9 min so far)
  fold 4: 19 images done (41.3 min so far)

=== XAI on 107 out-of-fold images (unseen patients) ===
iou_* higher = more stable | deletion LOWER = more faithful | insertion HIGHER | sanity_* LOWER
           method  deletion_auc  insertion_auc  iou_low  iou_mid  iou_high  sanity_iou_full  sanity_iou_mamba  map_std  spearman_vs_gradient
influence_rollout        0.6356         0.6178   0.7015   0.6122    0.4721           0.6346            0.6347   0.0674                0.0341
     influence_cc        0.5692         0.6568   0.4802   0.3675    0.2708           0.1363            0.1289   0.1570               -0.3850
influence_cc_last        0.5683         0.6577   0.4802   0.3675    0.2708           0.1298            0.1252   0.1827               -0.3856
 influence_cc_abs        0.5873         0.6374   0.4569   0.3559    0

In [18]:
from scipy.stats import wilcoxon, rankdata

def rank_biserial(d):
    d = d[d != 0]
    if len(d) == 0: return 0.0
    r = rankdata(np.abs(d)); wp, wm = r[d>0].sum(), r[d<0].sum()
    return float((wp-wm)/(wp+wm)) if (wp+wm) > 0 else 0.0

def boot_ci(d, n=N_BOOTSTRAP, alpha=0.05):
    rng = np.random.default_rng(SEED)
    means = d[rng.integers(0, len(d), size=(n, len(d)))].mean(1)
    return float(np.percentile(means, 100*alpha/2)), float(np.percentile(means, 100*(1-alpha/2)))

def holm(pvals):
    p = np.asarray(pvals, float); n = len(p)
    adj = np.empty(n); run = 0.0
    for rank, i in enumerate(np.argsort(p)):
        run = max(run, (n-rank)*p[i]); adj[i] = min(run, 1.0)
    return adj

stat_rows = []
for metric in METRICS:
    others = [m for m in METHODS if m != REFERENCE]
    raw = []
    for other in others:
        a, b = per_image[REFERENCE][metric], per_image[other][metric]
        d = a - b
        try: _, p = wilcoxon(a, b, zero_method="zsplit")
        except ValueError: p = 1.0
        lo_ci, hi_ci = boot_ci(d)
        raw.append(dict(metric=metric, reference=REFERENCE, other=other,
                        ref_mean=round(float(a.mean()),4), other_mean=round(float(b.mean()),4),
                        mean_diff=round(float(d.mean()),4),
                        ci95_low=round(lo_ci,4), ci95_high=round(hi_ci,4),
                        effect_size=round(rank_biserial(d),3), p_raw=float(p),
                        favours="reference" if (d.mean()>0)==HIGHER_IS_BETTER[metric] else other))
    for r, padj in zip(raw, holm([r["p_raw"] for r in raw])):
        r["p_holm"] = float(f"{padj:.3g}"); r["p_raw"] = float(f"{r['p_raw']:.3g}")
        r["significant"] = bool(padj < ALPHA)
        stat_rows.append(r)

stats_df = pd.DataFrame(stat_rows)
stats_df.to_csv(f"{OUT_DIR}/table5_significance.csv", index=False)

for metric in METRICS:
    sub = stats_df[stats_df.metric == metric]
    print(f"\n=== {metric}  ({'higher' if HIGHER_IS_BETTER[metric] else 'LOWER'} is better) ===")
    print(sub[["other","ref_mean","other_mean","mean_diff","ci95_low","ci95_high",
               "effect_size","p_holm","significant","favours"]].to_string(index=False))


=== deletion_auc  (LOWER is better) ===
            other  ref_mean  other_mean  mean_diff  ci95_low  ci95_high  effect_size       p_holm  significant           favours
influence_rollout    0.5692      0.6356    -0.0664   -0.0832    -0.0489       -0.821 9.840000e-13         True         reference
influence_cc_last    0.5692      0.5683     0.0009    0.0003     0.0015        0.310 6.090000e-03         True influence_cc_last
 influence_cc_abs    0.5692      0.5873    -0.0180   -0.0265    -0.0105       -0.694 1.330000e-08         True         reference
         grad_cam    0.5692      0.6345    -0.0653   -0.0810    -0.0500       -0.799 3.740000e-12         True         reference
             lime    0.5692      0.5887    -0.0195   -0.0376     0.0002       -0.336 5.090000e-03         True         reference
        mamba_lrp    0.5692      0.6372    -0.0679   -0.0852    -0.0512       -0.784 7.710000e-12         True         reference

=== insertion_auc  (higher is better) ===
            o

In [19]:
print("="*80)
print(f"VERDICT  (reference = {REFERENCE}, alpha = {ALPHA}, Holm-corrected within each metric)")
print("="*80)
for metric in METRICS:
    print(f"\n{metric}  ({'higher' if HIGHER_IS_BETTER[metric] else 'lower'} is better)")
    for _, r in stats_df[stats_df.metric == metric].iterrows():
        rng = f"95% CI [{r.ci95_low:+.4f}, {r.ci95_high:+.4f}]"
        if not r.significant:
            print(f"   vs {r.other:20s} NO detectable difference "
                  f"(diff {r.mean_diff:+.4f}, {rng}, p = {r.p_holm:.3g})")
        else:
            who = REFERENCE if r.favours == "reference" else r.other
            print(f"   vs {r.other:20s} {who} is BETTER "
                  f"(diff {r.mean_diff:+.4f}, {rng}, r = {r.effect_size:+.3f}, p = {r.p_holm:.3g})")
if QUICK_TEST:
    print("\n(QUICK_TEST: with so few images the smallest possible Holm-corrected p is large, "
          "so ignore these p-values. They become meaningful at PER_CLASS = 40.)")

VERDICT  (reference = influence_cc, alpha = 0.05, Holm-corrected within each metric)

deletion_auc  (lower is better)
   vs influence_rollout    influence_cc is BETTER (diff -0.0664, 95% CI [-0.0832, -0.0489], r = -0.821, p = 9.84e-13)
   vs influence_cc_last    influence_cc_last is BETTER (diff +0.0009, 95% CI [+0.0003, +0.0015], r = +0.310, p = 0.00609)
   vs influence_cc_abs     influence_cc is BETTER (diff -0.0180, 95% CI [-0.0265, -0.0105], r = -0.694, p = 1.33e-08)
   vs grad_cam             influence_cc is BETTER (diff -0.0653, 95% CI [-0.0810, -0.0500], r = -0.799, p = 3.74e-12)
   vs lime                 influence_cc is BETTER (diff -0.0195, 95% CI [-0.0376, +0.0002], r = -0.336, p = 0.00509)
   vs mamba_lrp            influence_cc is BETTER (diff -0.0679, 95% CI [-0.0852, -0.0512], r = -0.784, p = 7.71e-12)

insertion_auc  (higher is better)
   vs influence_rollout    influence_cc is BETTER (diff +0.0390, 95% CI [+0.0222, +0.0548], r = +0.650, p = 2.09e-08)
   vs influence_cc

## Section 10 — Figures

In [20]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
for name, (d, ii) in CURVES.items():
    t = np.linspace(0, 1, len(d))
    axs[0].plot(t, d, marker="o", ms=3, label=name); axs[1].plot(t, ii, marker="o", ms=3, label=name)
axs[0].set_title(f"Deletion, lower is better (n={n_img})"); axs[0].set_xlabel("fraction removed")
axs[1].set_title(f"Insertion, higher is better (n={n_img})"); axs[1].set_xlabel("fraction revealed")
for a in axs: a.set_ylabel("p(predicted class)"); a.legend(fontsize=8); a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig1_faithfulness.png", dpi=150); plt.close()

# accuracy per fold, per model
fig, ax = plt.subplots(figsize=(7, 4))
for name in MODELS:
    fm = fold_df[fold_df.model == name]
    ax.plot(fm.fold, fm.acc, marker="o", label=f"{name} ({fm.acc.mean():.3f} ± {fm.acc.std(ddof=1):.3f})")
ax.set_xlabel("fold"); ax.set_ylabel("test accuracy"); ax.set_title("Patient-level 5-fold CV")
ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig2_cv_accuracy.png", dpi=150); plt.close()

# qualitative maps: first image of each class
def denorm(t):
    m = torch.tensor(MEAN).view(3, 1, 1); s = torch.tensor(STD).view(3, 1, 1)
    return (t.cpu()*s + m).clamp(0, 1).permute(1, 2, 0).numpy()
show = []
for c in LH_CLASSES:
    k = next((i for i, cc in enumerate(CLS_OF_IMG) if cc == c), None)
    if k is not None: show.append(k)
fig, axes = plt.subplots(len(show), len(METHODS)+1, figsize=(3*(len(METHODS)+1), 3*len(show)))
axes = np.atleast_2d(axes)
for r, k in enumerate(show):
    j = IMG_ORDER[k]; f = int(xai_fold[list(xai_idx).index(j)])
    model = load_fold_model(f); x = center_crop_tensor(j)[None].to(DEVICE)
    with torch.no_grad(): pr = _logits(model, x).argmax(1)
    axes[r, 0].imshow(denorm(x[0])); axes[r, 0].set_ylabel(f"{CLS_OF_IMG[k]} (pred {LH_CLASSES[int(pr)]})", fontsize=8)
    for c, (nm, fn) in enumerate(METHODS.items(), start=1):
        a = fn(model, x, pr)[0, 0].detach().cpu().numpy()
        axes[r, c].imshow(denorm(x[0])); axes[r, c].imshow(a, cmap="jet", alpha=0.45)
        if r == 0: axes[r, c].set_title(nm, fontsize=8)
    del model
for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig3_maps.png", dpi=150); plt.close()

print("saved:")
for fn_ in sorted(os.listdir(OUT_DIR)): print("  ", fn_)

saved:
   cm_cnn_mamba_lcpre_oof.npy
   fig1_faithfulness.png
   fig2_cv_accuracy.png
   fig3_maps.png
   oof_predictions.csv
   per_image_xai_metrics.csv
   table5_significance.csv
   tableA_dataset_summary.csv
   tableB_official_split.csv
   tableC_cv_per_fold.csv
   tableD_cv_summary.csv
   tableE_model_differences.csv
   tableF_accuracy_by_grade.csv
   tableG_xai_comparison.csv
   tableH_xai_per_class.csv
   tableI_prediction_flip_rate.csv
   weights_cnn_mamba_lc25000_pretrain.pt
   weights_cnn_mamba_lcpre_fold0.pt
   weights_cnn_mamba_lcpre_fold1.pt
   weights_cnn_mamba_lcpre_fold2.pt
   weights_cnn_mamba_lcpre_fold3.pt
   weights_cnn_mamba_lcpre_fold4.pt


## Section 11 — How to read and report the results

### Files

| file | what it is |
|---|---|
| `tableA_dataset_summary.csv` | images and patients per class and grade |
| `tableB_official_split.csv` | the authors' split, comparable with their ResNet-50 (acc 0.90 at 20x) |
| `tableC_cv_per_fold.csv`, `tableD_cv_summary.csv` | **main accuracy result**, mean ± std and pooled with patient-bootstrap CI |
| `tableE_model_differences.csv` | ours vs scratch vs ResNet-50, paired, patient-bootstrap CI |
| `tableF_accuracy_by_grade.csv` | accuracy per differentiation grade |
| `tableG_xai_comparison.csv`, `table5_significance.csv` | XAI means and paired tests |
| `tableH_xai_per_class.csv`, `tableI_prediction_flip_rate.csv` | XAI per class, flip rate |
| `oof_predictions.csv` | every out-of-fold prediction, for any later analysis |

### Rules for honest reporting

* **Quote the CV result as the main accuracy**, with its std and CI. The official split has
  only a few test patients, so treat it as a comparison with the paper, not as the headline.
* **The CI is wide on purpose.** About 38 patients is small. A wide interval is the true
  uncertainty, not a flaw in the method.
* If `cnn_mamba_lcpre` does not beat `resnet50_imagenet`, say so. ResNet-50 has about 10x more
  parameters and was pre-trained on 1.2 million ImageNet images. Our model's contribution is the
  explanation method, not the accuracy.
* **Poorly differentiated tumours** are expected to be harder: they lose the typical glandular
  (aca) or keratinising (scc) pattern that separates the two cancers. Check `tableF`.
* Compare the XAI verdict with the LC25000 notebook. If `influence_cc` keeps its advantage on
  unseen patients from a second hospital, that is the strongest evidence in the thesis. If it
  does not, report where it changes.